# California Housing Regression with LightGBM and SHAP

This notebook loads the California Housing dataset, trains a LightGBM regression model, and uses SHAP to interpret the predictions.
The goal is to show key features, correlations, model performance, and shap-based explanations in a clean, reproducible workflow.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from lightgbm import LGBMRegressor
import shap

## Load Dataset

Load the California Housing dataset and convert it into a pandas DataFrame with feature names and a target column named `MedHouseVal`.

In [ ]:
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename("MedHouseVal")], axis=1)
df.head()

## Exploratory Data Analysis

Inspect the dataset structure, summary statistics, and feature correlations to understand the relationships before training the model.

In [ ]:
print("Dataset shape:", df.shape)
print()
print("Summary statistics:")
display(df.describe())

corr = df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

strong_corr = corr['MedHouseVal'].drop('MedHouseVal').abs().sort_values(ascending=False)
print("Features strongly correlated with MedHouseVal:")
display(strong_corr.head(10))

print("Important features summary:")
print("- MedInc often has the strongest positive relationship with house value.")
print("- AveRooms tends to increase value, but very high room counts may show diminishing returns.")
print("- HouseAge can have a mixed effect, where older homes may be worth less unless location and amenities compensate.")

## Train-Test Split

Split the data into training and testing sets using an 80-20 split with a fixed random state for reproducibility.

In [ ]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)

## Model Training (LightGBM)

Train a LightGBM regression model with reasonable default parameters for the California Housing dataset.

In [ ]:
model = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=7, random_state=42, verbose=-1)
model.fit(X_train, y_train)
print("Model training complete.")

## Model Evaluation

Evaluate the trained model using RMSE and R² score on the test set.

In [ ]:
predictions = model.predict(X_test)
rmse = mean_squared_error(y_test, predictions, squared=False)
r2 = r2_score(y_test, predictions)
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R2 score: {r2:.4f}")

## SHAP Analysis

Use SHAP to explain model predictions for the test set and compute interaction values.

In [ ]:
explainer = shap.Explainer(model, X_test)
shap_values = explainer(X_test)
interaction_values = shap.TreeExplainer(model).shap_interaction_values(X_test)
print("SHAP explanation generated for the test set.")

## SHAP Summary Plots

Visualize global feature importance and the overall distribution of SHAP values.

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title('SHAP Summary - Feature Importance')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Summary - Beeswarm Plot')
plt.tight_layout()
plt.show()

shap_mean_abs = np.abs(shap_values.values).mean(axis=0)
feature_importance = pd.Series(shap_mean_abs, index=X_test.columns).sort_values(ascending=False)
print("Top SHAP-important features:")
display(feature_importance.head(10))
print("")
print("MedInc is the most impactful feature for predicting house prices, followed by room count and location-related values.")

## SHAP Dependence Plots

Analyze non-linear effects and interactions for key features such as `MedInc`, `AveRooms`, and `HouseAge`.

In [ ]:
plt.figure(figsize=(10, 6))
shap.dependence_plot('MedInc', shap_values.values, X_test, show=False)
plt.title('SHAP Dependence: MedInc')
plt.xlabel('MedInc')
plt.ylabel('SHAP value')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.dependence_plot('AveRooms', shap_values.values, X_test, show=False)
plt.title('SHAP Dependence: AveRooms')
plt.xlabel('AveRooms')
plt.ylabel('SHAP value')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.dependence_plot('HouseAge', shap_values.values, X_test, show=False)
plt.title('SHAP Dependence: HouseAge')
plt.xlabel('HouseAge')
plt.ylabel('SHAP value')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.dependence_plot('MedInc', interaction_values, X_test, interaction_index='AveRooms', show=False)
plt.title('SHAP Interaction: MedInc vs AveRooms')
plt.xlabel('MedInc')
plt.ylabel('SHAP value')
plt.tight_layout()
plt.show()

## Partial Dependence Plots

Compare SHAP results with partial dependence plots to verify how feature values affect model predictions.

In [ ]:
features = ['MedInc', 'AveRooms', 'HouseAge', ('MedInc', 'AveRooms')]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
PartialDependenceDisplay.from_estimator(model, X_test, features, kind='average', ax=axes, grid_resolution=50)
fig.suptitle('Partial Dependence Plots')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Final Insights

Summarize the main findings from the model evaluation and SHAP analysis.

- The LightGBM model shows strong predictive performance for California housing values.
- SHAP identified `MedInc` as the most important feature, with non-linear effects and saturation at high income levels.
- `AveRooms` and `HouseAge` also influence predictions in non-linear ways, consistent with diminishing returns and age-related depreciation.
- SHAP interaction plots reveal how income and room count jointly impact value, while partial dependence plots confirm the overall shape of each feature effect.
- In real-world housing prediction, non-linear relationships are important because price effects depend on thresholds, feature combinations, and changing marginal influence.